In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/retinal-oct-c8/README.md
/kaggle/input/retinal-oct-c8/RetinalOCT_Dataset/RetinalOCT_Dataset/val/DR/dr_val_1127.jpg
/kaggle/input/retinal-oct-c8/RetinalOCT_Dataset/RetinalOCT_Dataset/val/DR/dr_val_1076.jpg
/kaggle/input/retinal-oct-c8/RetinalOCT_Dataset/RetinalOCT_Dataset/val/DR/dr_val_1059.jpg
/kaggle/input/retinal-oct-c8/RetinalOCT_Dataset/RetinalOCT_Dataset/val/DR/dr_val_1084.jpg
/kaggle/input/retinal-oct-c8/RetinalOCT_Dataset/RetinalOCT_Dataset/val/DR/dr_val_1193.jpg
/kaggle/input/retinal-oct-c8/RetinalOCT_Dataset/RetinalOCT_Dataset/val/DR/dr_val_1261.jpg
/kaggle/input/retinal-oct-c8/RetinalOCT_Dataset/RetinalOCT_Dataset/val/DR/dr_val_1132.jpg
/kaggle/input/retinal-oct-c8/RetinalOCT_Dataset/RetinalOCT_Dataset/val/DR/dr_val_1060.jpg
/kaggle/input/retinal-oct-c8/RetinalOCT_Dataset/RetinalOCT_Dataset/val/DR/dr_val_1080.jpg
/kaggle/input/retinal-oct-c8/RetinalOCT_Dataset/RetinalOCT_Dataset/val/DR/dr_val_1281.jpg
/kaggle/input/retinal-oct-c8/RetinalOCT_Dataset/RetinalOCT_Da

In [2]:
import tensorflow as tf
import numpy as np
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import AdamW

2026-03-02 07:00:30.258706: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772434830.713656      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772434830.811296      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772434831.780328      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772434831.780376      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772434831.780380      55 computation_placer.cc:177] computation placer alr

In [3]:
IMG_SIZE = 224
BATCH_SIZE = 128
EPOCHS = 100
INITIAL_LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_CLASSES = 8  # change accordingly

In [4]:
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np

train_path = "/kaggle/input/retinal-oct-c8/RetinalOCT_Dataset/RetinalOCT_Dataset/train"
val_path = "/kaggle/input/retinal-oct-c8/RetinalOCT_Dataset/RetinalOCT_Dataset/val"
test_path = "/kaggle/input/retinal-oct-c8/RetinalOCT_Dataset/RetinalOCT_Dataset/test"

#min–max normalization.

def minmax_normalize(image, label):
    image = tf.cast(image, tf.float32)
    min_val = tf.reduce_min(image)
    max_val = tf.reduce_max(image)
    image = (image - min_val) / (max_val - min_val + 1e-7)
    return image, label

#medically safe augmentation (no shape tricks).

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomCrop(200, 200),
    tf.keras.layers.Resizing(224, 224)])

#gamma manually.

def random_gamma(image, label):
    gamma = tf.random.uniform([], 0.8, 1.2)
    image = tf.image.adjust_gamma(image, gamma)
    return image, label


train_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    image_size=(224, 224),
    batch_size=None
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_path,
    image_size=(224, 224),
    batch_size=128
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_path,
    image_size=(224, 224),
    batch_size=128,
    shuffle=False
)

# Min-max normalization
train_ds = train_ds.map(minmax_normalize)
val_ds = val_ds.map(minmax_normalize)
test_ds = test_ds.map(minmax_normalize)

# Augmentation ONLY to training
train_ds = train_ds.map(random_gamma)
train_ds = train_ds.map(lambda x, y: (data_augmentation(x), y))
train_ds = train_ds.batch(128).prefetch(tf.data.AUTOTUNE)

val_ds = val_ds.prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.prefetch(tf.data.AUTOTUNE)

I0000 00:00:1772434866.382795      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Found 18400 files belonging to 8 classes.
Found 2800 files belonging to 8 classes.
Found 2800 files belonging to 8 classes.


In [5]:
NUM_CLASSES = 8

In [20]:
def build_resnet50():
    base_model = tf.keras.applications.ResNet50(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    
    base_model.trainable = False  # Stage 1
    
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(NUM_CLASSES,activation='softmax')(x)  # logits
    
    model = tf.keras.Model(inputs, outputs)
    return model, base_model


In [23]:
from tensorflow.keras.optimizers import AdamW
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)

def train_resnet():
    model, base_model = build_resnet50()

    # -------- Stage 1: Train classifier head --------
    optimizer_stage1 = AdamW(
        learning_rate=INITIAL_LR,
        weight_decay=WEIGHT_DECAY
    )

    model.compile(
        optimizer=optimizer_stage1,
        loss=loss_fn,
        metrics=["accuracy"]
    )

    model.summary()

    print("Stage 1 Training...")
    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=10
    )

    # -------- Stage 2: Fine-tune full network --------
    base_model.trainable = True

    optimizer_stage2 = AdamW(
        learning_rate=1e-4,
        weight_decay=WEIGHT_DECAY
    )

    model.compile(
        optimizer=optimizer_stage2,
        loss=loss_fn,
        metrics=["accuracy"]
    )

    print("Stage 2 Fine-tuning...")
    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=90,
        callbacks=[
            tf.keras.callbacks.ReduceLROnPlateau(
                monitor='val_loss',
                factor=0.5,
                patience=5,
                verbose=1
            ),
            tf.keras.callbacks.EarlyStopping(
                monitor='val_loss',
                patience=10,
                restore_best_weights=True
            )
        ]
    )

    return model


In [24]:
from sklearn.metrics import classification_report, confusion_matrix

model = train_resnet()

test_loss, test_acc = model.evaluate(test_ds)
print("Test Accuracy:", test_acc)

y_true = []
y_pred = []

for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    preds = tf.argmax(preds, axis=1)
    y_true.extend(labels.numpy())
    y_pred.extend(preds.numpy())

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, digits=4))

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_9 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_7      │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 8)              │        16,392 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,604,104 (90.04 MB)

 Trainable params: 16,392 (64.03 KB)

 Non-trainable params: 23,587,712 (89.98 MB)

Stage 1 Training...
Epoch 1/10
144/144 ━━━━━━━━━━━━━━━━━━━━ 163s 1s/step - accuracy: 0.1358 - loss: 2.2370 - val_accuracy: 0.2075 - val_loss: 2.0152
Epoch 2/10
144/144 ━━━━━━━━━━━━━━━━━━━━ 138s 955ms/step - accuracy: 0.1640 - loss: 2.0573 - val_accuracy: 0.2543 - val_loss: 1.9582
Epoch 3/10
144/144 ━━━━━━━━━━━━━━━━━━━━ 138s 958ms/step - accuracy: 0.1823 - loss: 2.0250 - val_accuracy: 0.3325 - val_loss: 1.9190
Epoch 4/10
144/144 ━━━━━━━━━━━━━━━━━━━━ 138s 959ms/step - accuracy: 0.2179 - loss: 1.9898 - val_accuracy: 0.3475 - val_loss: 1.8841
Epoch 5/10
144/144 ━━━━━━━━━━━━━━━━━━━━ 138s 957ms/step - accuracy: 0.2304 - loss: 1.9703 - val_accuracy: 0.3318 - val_loss: 1.8520
Epoch 6/10
144/144 ━━━━━━━━━━━━━━━━━━━━ 138s 957ms/step - accuracy: 0.2408 - loss: 1.9489 - val_accuracy: 0.3989 - val_loss: 1.8263
Epoch 7/10
144/144 ━━━━━━━━━━━━━━━━━━━━ 139s 962ms/step - accuracy: 0.2587 - loss: 1.9314 - val_accuracy: 0.3796 - val_loss: 1.8062
Epoch 8/10
144/144 ━━━━━━━━━━━━━━━━━━━━ 138s 959ms/step - a

2026-03-02 07:40:49.647147: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng0{} for conv %cudnn-conv-bw-input.24 = (f32[128,1024,14,14]{3,2,1,0}, u8[0]{0}) custom-call(f32[128,2048,7,7]{3,2,1,0} %bitcast.75239, f32[2048,1024,1,1]{3,2,1,0} %bitcast.56140), window={size=1x1 stride=2x2}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBackwardInput", metadata={op_type="Conv2DBackpropInput" op_name="gradient_tape/functional_3_1/resnet50_1/conv5_block1_0_conv_1/convolution/Conv2DBackpropInput" source_file="/usr/local/lib/python3.12/dist-packages/tensorflow/python/framework/ops.py" source_line=1200}, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"conv_result_scale":1,"activation_mode":"kNone","side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false} is taking a while...
2026-03-02 07:40:50.577345: E external/local_xla/xla/service/slow_operation_alarm.cc:140] The operation took 

143/144 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step - accuracy: 0.4158 - loss: 5.7554

2026-03-02 07:44:20.713426: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng0{} for conv %cudnn-conv-bw-input.24 = (f32[96,1024,14,14]{3,2,1,0}, u8[0]{0}) custom-call(f32[96,2048,7,7]{3,2,1,0} %bitcast.75239, f32[2048,1024,1,1]{3,2,1,0} %bitcast.56140), window={size=1x1 stride=2x2}, dim_labels=bf01_oi01->bf01, custom_call_target="__cudnn$convBackwardInput", metadata={op_type="Conv2DBackpropInput" op_name="gradient_tape/functional_3_1/resnet50_1/conv5_block1_0_conv_1/convolution/Conv2DBackpropInput" source_file="/usr/local/lib/python3.12/dist-packages/tensorflow/python/framework/ops.py" source_line=1200}, backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"conv_result_scale":1,"activation_mode":"kNone","side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false} is taking a while...
2026-03-02 07:44:21.120102: E external/local_xla/xla/service/slow_operation_alarm.cc:140] The operation took 1.

144/144 ━━━━━━━━━━━━━━━━━━━━ 284s 2s/step - accuracy: 0.4182 - loss: 5.7045 - val_accuracy: 0.1250 - val_loss: 2.4181 - learning_rate: 1.0000e-04
Epoch 2/90
144/144 ━━━━━━━━━━━━━━━━━━━━ 185s 1s/step - accuracy: 0.9056 - loss: 0.2903 - val_accuracy: 0.1250 - val_loss: 3.5564 - learning_rate: 1.0000e-04
Epoch 3/90
144/144 ━━━━━━━━━━━━━━━━━━━━ 184s 1s/step - accuracy: 0.9524 - loss: 0.1476 - val_accuracy: 0.1250 - val_loss: 5.7536 - learning_rate: 1.0000e-04
Epoch 4/90
144/144 ━━━━━━━━━━━━━━━━━━━━ 184s 1s/step - accuracy: 0.9649 - loss: 0.1086 - val_accuracy: 0.1250 - val_loss: 5.1274 - learning_rate: 1.0000e-04
Epoch 5/90
144/144 ━━━━━━━━━━━━━━━━━━━━ 184s 1s/step - accuracy: 0.9699 - loss: 0.0993 - val_accuracy: 0.1389 - val_loss: 6.5813 - learning_rate: 1.0000e-04
Epoch 6/90
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.9728 - loss: 0.0787
Epoch 6: ReduceLROnPlateau reducing learning rate to 4.999999873689376e-05.
144/144 ━━━━━━━━━━━━━━━━━━━━ 185s 1s/step - accuracy: 0.9728 - lo

In [31]:
import tensorflow as tf
import gc

del model
tf.keras.backend.clear_session()
gc.collect()

0

In [33]:
from tensorflow.keras.utils import register_keras_serializable
import tensorflow as tf

#SE BLOCK

@register_keras_serializable()
def se_block(x, reduction_ratio=16):
    channels = tf.keras.backend.int_shape(x)[-1]
    se = tf.keras.layers.GlobalAveragePooling2D()(x)
    se = tf.keras.layers.Dense(max(channels // reduction_ratio, 1), activation='relu')(se)
    se = tf.keras.layers.Dense(channels, activation='sigmoid')(se)
    se = tf.keras.layers.Reshape((1, 1, channels))(se)
    return tf.keras.layers.Multiply()([x, se])


#FINE BRANCH

def fine_branch(x, filters):
    shortcut = tf.keras.layers.Conv2D(filters, 1, padding='same', use_bias=False)(x)
    shortcut = tf.keras.layers.BatchNormalization()(shortcut)

    x = tf.keras.layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)

    x = tf.keras.layers.Conv2D(filters, 3, padding='same', use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)

    x = tf.keras.layers.Add()([x, shortcut])
    x = tf.keras.layers.ReLU()(x)

    return x


#COARSE BRANCH

def coarse_branch(x, filters):
    shortcut = tf.keras.layers.Conv2D(filters, 1, padding='same', use_bias=False)(x)
    shortcut = tf.keras.layers.BatchNormalization()(shortcut)

    # Depthwise + dilated conv for efficient large receptive field
    x = tf.keras.layers.DepthwiseConv2D(3, padding='same', dilation_rate=2, use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)

    x = tf.keras.layers.Conv2D(filters, 1, padding='same', use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)

    x = tf.keras.layers.Add()([x, shortcut])
    x = tf.keras.layers.ReLU()(x)

    return x


#FUSION BLOCK

def fine_coarse_fusion(x, filters):
    fine = fine_branch(x, filters)
    coarse = coarse_branch(x, filters)

    # Addition
    x = tf.keras.layers.Add()([fine, coarse])

    # Channel recalibration
    x = se_block(x)
    return x

#MODEL

def build_finecoarsemodel(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=NUM_CLASSES):
    inputs = tf.keras.Input(shape=input_shape)

    #early downsampling
    x = tf.keras.layers.Conv2D(32, 3, strides=2, padding='same', use_bias=False)(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.ReLU()(x)

    #Stack 1
    x = fine_coarse_fusion(x, 32)
    x = tf.keras.layers.MaxPooling2D()(x)

    #Stack 2
    x = fine_coarse_fusion(x, 64)
    x = tf.keras.layers.MaxPooling2D()(x)

    #Stack 3
    x = fine_coarse_fusion(x, 128)
    x = tf.keras.layers.MaxPooling2D()(x)

    #Head
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dropout(0.3)(x)

    outputs = tf.keras.layers.Dense(num_classes,activation='softmax')(x)
    return tf.keras.Model(inputs, outputs)

In [34]:
from tensorflow.keras.optimizers import AdamW

In [35]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)

checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    "best_model.keras",
    monitor="val_loss",
    save_best_only=True,
    verbose=1
)

early_stop = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)
reduce_lr = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

def train_fine_coarse():
    fc_model = build_finecoarsemodel(input_shape=(224,224,3), num_classes=NUM_CLASSES)

    optimizer = AdamW(learning_rate=INITIAL_LR, weight_decay=WEIGHT_DECAY)

    fc_model.compile(
        optimizer=optimizer,
        loss=loss_fn,
        metrics=["accuracy"]
    )

    fc_model.summary()

    history = fc_model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=[checkpoint_cb, early_stop, reduce_lr])

    return fc_model

modelfcfusion = train_fine_coarse()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 112, 112,  │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu (ReLU)        │ (None, 112, 112,  │          0 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 112, 112,  │      9,216 │ re_lu[0][0]       │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ depthwise_conv2d    │ (None, 112, 112,  │        288 │ re_lu[0][0]       │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        128 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        128 │ depthwise_conv2d… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_1 (ReLU)      │ (None, 112, 112,  │          0 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_3 (ReLU)      │ (None, 112, 112,  │          0 │ batch_normalizat… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 112, 112,  │      9,216 │ re_lu_1[0][0]     │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 112, 112,  │      1,024 │ re_lu[0][0]       │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 112, 112,  │      1,024 │ re_lu_3[0][0]     │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 112, 112,  │      1,024 │ re_lu[0][0]       │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        128 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        128 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        128 │ conv2d_5[0][0]  

 Total params: 373,846 (1.43 MB)

 Trainable params: 371,286 (1.42 MB)

 Non-trainable params: 2,560 (10.00 KB)

Epoch 1/100
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.3987 - loss: 1.3846   
Epoch 1: val_loss improved from inf to 4.89064, saving model to best_model.keras
144/144 ━━━━━━━━━━━━━━━━━━━━ 179s 1s/step - accuracy: 0.3994 - loss: 1.3827 - val_accuracy: 0.1250 - val_loss: 4.8906 - learning_rate: 0.0010
Epoch 2/100
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 993ms/step - accuracy: 0.6816 - loss: 0.7465
Epoch 2: val_loss did not improve from 4.89064
144/144 ━━━━━━━━━━━━━━━━━━━━ 147s 1s/step - accuracy: 0.6818 - loss: 0.7461 - val_accuracy: 0.1250 - val_loss: 7.8347 - learning_rate: 0.0010
Epoch 3/100
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.7895 - loss: 0.5416
Epoch 3: val_loss did not improve from 4.89064
144/144 ━━━━━━━━━━━━━━━━━━━━ 148s 1s/step - accuracy: 0.7896 - loss: 0.5413 - val_accuracy: 0.1250 - val_loss: 6.2820 - learning_rate: 0.0010
Epoch 4/100
144/144 ━━━━━━━━━━━━━━━━━━━━ 0s 988ms/step - accuracy: 0.8437 - loss: 0.4048
Epoch 4: val_loss improved from 4.89064 to 0.9

In [38]:
from sklearn.metrics import classification_report, confusion_matrix

test_loss, test_acc = modelfcfusion.evaluate(test_ds)
print("Fine-Coarse Test Accuracy:", test_acc)

y_true = []
y_pred = []

for images, labels in test_ds:
    preds = modelfcfusion.predict(images, verbose=0)
    preds = tf.argmax(preds, axis=1)
    y_true.extend(labels.numpy())
    y_pred.extend(preds.numpy())

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(y_true, y_pred, digits=4))

22/22 ━━━━━━━━━━━━━━━━━━━━ 3s 151ms/step - accuracy: 0.9418 - loss: 0.1671
Fine-Coarse Test Accuracy: 0.9189285635948181
Confusion Matrix:
[[349   0   0   0   0   1   0   0]
 [  0 308   0   1   0  41   0   0]
 [  0   0 350   0   0   0   0   0]
 [  0  23   0 272   0  15   0  40]
 [  0   0   1   0 349   0   0   0]
 [  0   8   0   0   0 338   0   4]
 [  0   0   8   1  39   0 302   0]
 [  0   7   0   1   0  37   0 305]]

Classification Report:
              precision    recall  f1-score   support

           0     1.0000    0.9971    0.9986       350
           1     0.8902    0.8800    0.8851       350
           2     0.9749    1.0000    0.9873       350
           3     0.9891    0.7771    0.8704       350
           4     0.8995    0.9971    0.9458       350
           5     0.7824    0.9657    0.8645       350
           6     1.0000    0.8629    0.9264       350
           7     0.8739    0.8714    0.8727       350

    accuracy                         0.9189      2800
   macro avg  